# 🚀 Spotify Support DeBERTa-v3 Intent Classifier Fine-Tuning
**Track 1: High-Performance, Low-Latency Intent Classification**
- **Base Model**: `microsoft/deberta-v3-small` (86M parameters)
- **Target**: Jump from 51.5% baseline to 88–92% accuracy across 8 customer support intents
- **Runtime**: Free Google Colab T4 GPU (~3 minutes training time)

### Step 1: Check GPU & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q transformers datasets accelerate sentencepiece tiktoken evaluate scikit-learn

### Step 2: Upload Data Bundle (`colab_bundle.zip`)
Run this cell and upload `colab_bundle.zip` (it contains `intent_train.jsonl`, `intent_val.jsonl`, and `golden_eval.jsonl`).

In [ ]:
from google.colab import files
import zipfile
import os

if not os.path.exists('colab_bundle.zip'):
    print('Please upload colab_bundle.zip:')
    uploaded = files.upload()

with zipfile.ZipFile('colab_bundle.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print('Data unzipped successfully:')
!ls -lh *.jsonl

### Step 3: Load Dataset & Tokenizer

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

MODEL_ID = 'microsoft/deberta-v3-small'
INTENT_MAP = {
    'content_availability': 0,
    'technical_support': 1,
    'account_access': 2,
    'billing_and_payments': 3,
    'subscription_and_plans': 4,
    'platform_and_regional': 5,
    'product_feedback': 6,
    'other_or_ambiguous': 7
}
ID_TO_LABEL = {v: k for k, v in INTENT_MAP.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

dataset = load_dataset('json', data_files={
    'train': 'intent_train.jsonl',
    'validation': 'intent_val.jsonl'
})
print(f'Train samples: {len(dataset["train"])}, Val samples: {len(dataset["validation"])}')

def tokenize_fn(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=128)
    enc['labels'] = [int(l) for l in batch['label']]
    return enc

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['text', 'label', 'example_id', 'intent', 'source'])

### Step 4: Fine-Tune DeBERTa-v3 on GPU (~3 minutes)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'macro_f1': macro_f1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(INTENT_MAP),
    id2label=ID_TO_LABEL,
    label2id=INTENT_MAP
)

training_args = TrainingArguments(
    output_dir='./deberta-spotify-checkpoints',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=False, # Stable FP32 on T4 GPU
    logging_steps=20,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model('./deberta-intent-spotify-best')
tokenizer.save_pretrained('./deberta-intent-spotify-best')
print('Training complete! Best model saved to ./deberta-intent-spotify-best')

### Step 5: Benchmark Against 200 Curated Gold Test Cases (`golden_eval.jsonl`)

In [ ]:
import json
import time
from sklearn.metrics import classification_report

gold_records = [json.loads(line) for line in open('golden_eval.jsonl', 'r', encoding='utf-8')]
texts = [r.get('message') or r.get('customer_message') for r in gold_records]
gold_labels = [INTENT_MAP[r['intent']] for r in gold_records]

model.eval()
predictions = []
batch_size = 32
start_time = time.perf_counter()

with torch.no_grad():
    for i in range(0, len(texts), batch_size):
        b_texts = texts[i:i + batch_size]
        inputs = tokenizer(b_texts, return_tensors='pt', truncation=True, max_length=128, padding=True).to(model.device)
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()
        predictions.extend(preds)

elapsed = time.perf_counter() - start_time
acc = accuracy_score(gold_labels, predictions)
macro_f1 = f1_score(gold_labels, predictions, average='macro', zero_division=0)

print('=' * 65)
print('DEBERTA INTENT CLASSIFIER - GOLD BENCHMARK RESULTS')
print('=' * 65)
print(f'Baseline Accuracy:     51.50%')
print(f'Fine-Tuned Accuracy:   {acc * 100:.2f}%')
print(f'Macro F1 Score:        {macro_f1:.4f}')
print(f'Latency per query:     {(elapsed / len(texts)) * 1000:.2f} ms')
print('=' * 65)
print('\nDetailed Per-Class Breakdown:')
target_names = [ID_TO_LABEL[i] for i in range(len(INTENT_MAP))]
print(classification_report(gold_labels, predictions, target_names=target_names, digits=4, zero_division=0))

### Step 6: Download the Fine-Tuned Model Weights to Your PC
Run this cell to zip and download `deberta-intent-spotify-best.zip` so you can use it locally in `E:/Reply_agent/models/`.

In [ ]:
!zip -r deberta-intent-spotify-best.zip deberta-intent-spotify-best
files.download('deberta-intent-spotify-best.zip')